# CA Exam Checker AI — Prepare 8 Evaluated Copies (May 2026)

This is a **CPU-safe preparation step** for `CA_Inter_May2026_Evaluated_Copies_Pack.zip`. It stages all 8 evaluated PDFs, extracts the selectable-text result-summary tables and any selectable step-mark labels, checks the awarded totals against question-level totals, and creates a human-review template in Drive.

**This does not train model weights.** The answer pages are primarily scanned handwriting and the current SmolLM2 supervisor is text-only. Extracted marks are candidate labels, not fully verified supervised examples. Each row needs the exact May 2026 question paper/Suggested Answer and reviewed answer evidence before training.

Run cells top to bottom. You upload **one ZIP only**; the PDFs inside are processed automatically.

In [ ]:
import os, sys, json, subprocess, hashlib
from pathlib import Path

from google.colab import drive, files
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluated_copies_may2026')
WORK_DIR.mkdir(parents=True, exist_ok=True)
INPUT_ZIP = WORK_DIR / 'CA_Inter_May2026_Evaluated_Copies_Pack.zip'

if INPUT_ZIP.is_file():
    print('✅ Reusing ZIP already in Drive:', INPUT_ZIP)
else:
    print('Upload the single file: CA_Inter_May2026_Evaluated_Copies_Pack.zip')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f'Expected exactly one ZIP file, received {len(uploaded)} files.')
    filename, data = next(iter(uploaded.items()))
    if not filename.lower().endswith('.zip'):
        raise ValueError('Please upload the evaluated-copies ZIP, not individual PDFs.')
    if len(data) > 250 * 1024 * 1024:
        raise ValueError('ZIP exceeds the 250 MB safety limit.')
    INPUT_ZIP.write_bytes(data)
    print('✅ ZIP saved to Drive:', INPUT_ZIP)

zip_sha256 = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
print('ZIP size (MB):', round(INPUT_ZIP.stat().st_size / (1024 * 1024), 2))
print('ZIP SHA256:', zip_sha256)

In [ ]:
# Install only the PDF parser needed for preparation (no GPU or training dependencies here).
try:
    import fitz
    print('✅ PyMuPDF already available:', fitz.VersionBind)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyMuPDF==1.26.4'], check=True)
    import fitz
    print('✅ PyMuPDF installed:', fitz.VersionBind)

import requests
SCRIPT_URL = 'https://raw.githubusercontent.com/thomasshelby-pixel/AI-Exam/main/research/ca-evaluation-supervisor-ai/prepare_evaluated_copies.py'
SCRIPT_PATH = Path('/content/prepare_evaluated_copies.py')
response = requests.get(SCRIPT_URL, timeout=30)
response.raise_for_status()
if 'def extract_question_rows' not in response.text or 'training_examples_emitted' not in response.text:
    raise RuntimeError('Downloaded preparation script failed its source sanity check.')
SCRIPT_PATH.write_text(response.text, encoding='utf-8')
print('✅ Preparation code loaded from repository:', SCRIPT_PATH)

In [ ]:
OUTPUT_DIR = WORK_DIR / 'prepared_review'
subprocess.run(
    [sys.executable, str(SCRIPT_PATH), '--zip', str(INPUT_ZIP), '--output-dir', str(OUTPUT_DIR), '--expected-pdf-count', '8'],
    check=True,
)
print('\n✅ Prepared review folder:', OUTPUT_DIR)

In [ ]:
from collections import Counter
manifest_path = OUTPUT_DIR / 'preparation_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
import csv
with (OUTPUT_DIR / 'copy_integrity_audit.csv').open(encoding='utf-8-sig', newline='') as f:
    audit = list(csv.DictReader(f))
print('STATUS:', manifest['status'])
print('Copies:', manifest['source_pdf_count'], '| Pages:', manifest['source_page_count'])
print('Question/sub-question score candidates:', manifest['structured_question_mark_candidates'])
print('Selectable step-mark labels:', manifest['step_mark_overlay_candidates'])
print('Training examples emitted:', manifest['training_examples_emitted'])
print('\nOverall-score consistency check:')
for row in audit:
    print(f"{row['subject']}: total={row['marks_awarded_total']}/{row['maximum_marks']} | question-row sum={row['sum_top_level_question_marks']} | {row['overall_total_check']}")
print('\nReview files:')
for name in ['question_mark_candidates.csv', 'step_mark_candidates.csv', 'human_review_template.csv', 'pdf_page_inventory.csv', 'copy_integrity_audit.csv', 'README_NEXT_STEPS.txt']:
    p = OUTPUT_DIR / name
    print(('✅' if p.exists() else '❌'), p)
print('\nNext: verify the matching May 2026 question paper and Suggested Answer, then visually review/transcribe answer evidence. Do not train directly from summary marks alone.')